# 🔧 Preprocessing & Feature Engineering
Based on EDA findings.

- Source name removal — "breitbart", "new york times" etc. were dominating fake news titles. If we leave them in, the model learns "this outlet = fake" rather than actual language patterns, which is cheating and won't generalise
- Article word count kept as a feature — your EDA showed a clear spike in fake articles under 100 words, making this one of your strongest handcrafted features
- Exclamation/caps kept but deprioritised — the EDA showed these were actually higher in real news (counterintuitive), so we include them but won't rely on them
- TF-IDF with bigrams — captures phrases like "breaking news" or "white house" rather than just single words
- Saves a .pkl file at the end with everything neatly packaged, so the modelling notebook can just load it and go

In [4]:
import pandas as pd
import numpy as np
import re
import sys
!{sys.executable} -m pip install nltk
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import pickle

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('Libraries loaded ✅')

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached regex-2026.5.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached regex-2026.5.9-cp312-cp312-macosx_11_0_arm64.whl (289 kB)


[nltk_data] Downloading package stopwords to /Users/shant/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/shant/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/shant/nltk_data...


Libraries loaded ✅


## 1. Load & Basic Clean

In [6]:
df = pd.read_csv('WELFake_Dataset.csv').drop(columns=['Unnamed: 0'])
df = df.dropna(subset=['title', 'text', 'label'])
df = df.drop_duplicates()
df['label'] = df['label'].astype(int)
print(f'Shape after cleaning: {df.shape}')
df.head(3)

Shape after cleaning: (63121, 3)


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0


## 2. Remove Leaky Source Names
> **EDA finding:** Fake news titles contained outlet names like 'breitbart', 'new york times'
> which could cause the model to cheat by memorising sources rather than learning language patterns.
> We strip these out to keep the classifier honest.

In [7]:
LEAKY_TERMS = [
    'breitbart', 'new york times', 'nyt', 'reuters', 'buzzfeed',
    'infowars', 'fox news', 'cnn', 'msnbc', 'washington post',
    'huffington post', 'huffpost', 'daily mail', 'guardian'
]

def remove_leaky_terms(text):
    text = text.lower()
    for term in LEAKY_TERMS:
        text = text.replace(term, '')
    return text

df['title'] = df['title'].astype(str).apply(remove_leaky_terms)
df['text']  = df['text'].astype(str).apply(remove_leaky_terms)
print('Leaky source names removed ✅')

Leaky source names removed ✅


## 3. Text Cleaning Function

In [8]:
lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'<.*?>', '', text)                 # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)              # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in STOPWORDS]
    return ' '.join(tokens)

print('Cleaning titles...')
df['title_clean'] = df['title'].apply(clean_text)
print('Cleaning article text...')
df['text_clean']  = df['text'].apply(clean_text)

# Combine title + text into one field for TF-IDF
df['combined_clean'] = df['title_clean'] + ' ' + df['text_clean']

print('Text cleaning done ✅')
df[['title_clean', 'text_clean', 'combined_clean']].head(3)

Cleaning titles...
Cleaning article text...
Text cleaning done ✅


,title_clean,text_clean,combined_clean
0,law enforcement high alert following threat co...,comment expected barack obama member fyf fukyo...,law enforcement high alert following threat co...
2,unbelievable obamas attorney general say charl...,demonstrator gathered last night exercising co...,unbelievable obamas attorney general say charl...
3,bobby jindal raised hindu us story christian c...,dozen politically active pastor came private d...,bobby jindal raised hindu us story christian c...


In [10]:
df.head(1)

,title,text,label,title_clean,text_clean,combined_clean
0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1,law enforcement high alert following threat co...,comment expected barack obama member fyf fukyo...,law enforcement high alert following threat co...


## 4. Handcrafted Features
> **EDA finding:** Article length was a strong signal (fake articles tend to be shorter).
> Exclamation marks / caps were higher in *real* news — counterintuitive but we still include
> them as features and let the model decide their weight.

In [11]:
def extract_features(df):
    feats = pd.DataFrame()

    # Length features — strong signal from EDA
    feats['title_word_count']   = df['title'].apply(lambda x: len(str(x).split()))
    feats['text_word_count']    = df['text'].apply(lambda x: len(str(x).split()))
    feats['title_char_count']   = df['title'].apply(lambda x: len(str(x)))
    feats['text_char_count']    = df['text'].apply(lambda x: len(str(x)))

    # Punctuation features — weaker signal but included
    feats['exclamation_count']  = df['title'].apply(lambda x: str(x).count('!'))
    feats['question_count']     = df['title'].apply(lambda x: str(x).count('?'))
    feats['caps_ratio']         = df['title'].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )

    # Extra signals
    feats['unique_word_ratio']  = df['text_clean'].apply(
        lambda x: len(set(str(x).split())) / max(len(str(x).split()), 1)
    )
    feats['avg_word_length']    = df['text_clean'].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )

    return feats

handcrafted_features = extract_features(df)
print(f'Handcrafted features shape: {handcrafted_features.shape}')
handcrafted_features.describe().round(3)

Handcrafted features shape: (63121, 9)


,title_word_count,text_word_count,title_char_count,text_char_count,exclamation_count,question_count,caps_ratio,unique_word_ratio,avg_word_length
count,63121.000,63121.000,63121.000,63121.000,63121.000,63121.000,63121.0,63121.000,63121.000
mean,11.623,545.899,73.276,3294.762,0.049,0.044,0.0,0.713,6.284
std,3.713,610.795,22.622,3637.574,0.248,0.215,0.0,0.123,0.804
min,1.000,0.000,1.000,1.000,0.000,0.000,0.0,0.000,0.000
25%,9.000,240.000,60.000,1453.000,0.000,0.000,0.0,0.652,6.111
50%,11.000,404.000,70.000,2462.000,0.000,0.000,0.0,0.712,6.348
75%,13.000,676.000,83.000,4109.000,0.000,0.000,0.0,0.776,6.577
max,72.000,24232.000,456.000,142958.000,9.000,5.000,0.0,1.000,42.591


## 5. Train / Test Split

In [12]:
X_text = df['combined_clean']
X_feats = handcrafted_features
y = df['label']

# Stratified split to preserve class balance
(
    X_text_train, X_text_test,
    X_feats_train, X_feats_test,
    y_train, y_test
) = train_test_split(
    X_text, X_feats, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train size : {len(y_train):,}')
print(f'Test size  : {len(y_test):,}')
print(f'Train class balance:\n{y_train.value_counts(normalize=True).round(3)}')

Train size : 50,496
Test size  : 12,625
Train class balance:
label
0    0.551
1    0.449
Name: proportion, dtype: float64


## 6. TF-IDF Vectorisation

In [13]:
tfidf = TfidfVectorizer(
    max_features=50000,   # top 50k terms
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # dampen very frequent terms
    min_df=3              # ignore very rare terms
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test  = tfidf.transform(X_text_test)

print(f'TF-IDF train shape: {X_tfidf_train.shape}')
print(f'TF-IDF test shape : {X_tfidf_test.shape}')

TF-IDF train shape: (50496, 50000)
TF-IDF test shape : (12625, 50000)


## 7. Combined Feature Matrix (TF-IDF + Handcrafted)
> Used for XGBoost — combines the TF-IDF sparse matrix with the handcrafted numeric features.

In [14]:
X_feats_train_sparse = csr_matrix(X_feats_train.values)
X_feats_test_sparse  = csr_matrix(X_feats_test.values)

X_combined_train = hstack([X_tfidf_train, X_feats_train_sparse])
X_combined_test  = hstack([X_tfidf_test,  X_feats_test_sparse])

print(f'Combined train shape: {X_combined_train.shape}')
print(f'Combined test shape : {X_combined_test.shape}')

Combined train shape: (50496, 50009)
Combined test shape : (12625, 50009)


## 8. Save Everything for Modelling Notebook

In [15]:
import pickle

artifacts = {
    # TF-IDF only (for Logistic Regression)
    'X_tfidf_train': X_tfidf_train,
    'X_tfidf_test' : X_tfidf_test,
    # Combined (for XGBoost)
    'X_combined_train': X_combined_train,
    'X_combined_test' : X_combined_test,
    # Raw text splits (for DistilBERT)
    'X_text_train': X_text_train.reset_index(drop=True),
    'X_text_test' : X_text_test.reset_index(drop=True),
    # Labels
    'y_train': y_train.reset_index(drop=True),
    'y_test' : y_test.reset_index(drop=True),
    # Vectoriser (needed to transform new data later)
    'tfidf': tfidf
}

with open('preprocessed_data.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('All artifacts saved to preprocessed_data.pkl ✅')
print('Ready for modelling notebook!')

All artifacts saved to preprocessed_data.pkl ✅
Ready for modelling notebook!
